# Speed: the fast Grassmann layer

The paper projects each offset's 496 Plücker numbers to 256 and then averages the six results. Projecting and averaging are both linear, so this notebook's code averages first and projects once. It also builds the average from the 32×32 pair products, so the slow upper-triangle lookup happens once instead of six times, and it takes the normaliser from Lagrange's identity, ‖a∧b‖² = ‖a‖²‖b‖² − (a·b)². Same maths, fewer operations.

1. **Same numbers?** Fast path vs the paper's order on this GPU, in float32: outputs and every gradient.
2. **How much faster?** Training tokens/s for each model, fast vs paper order.
3. **Where does the time go?** A profile of a few training steps, to explain the shared slowdown on the L4.

No data download is needed: speed is measured on random tokens. About 5 minutes on an A100.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
import os, json, subprocess, sys
REPO, DATA, OUTDIR = '/content/grassman', '/content/data', '/content/results'
RUNS = f'{OUTDIR}/speed.json'
os.makedirs(OUTDIR, exist_ok=True)
if os.path.exists(REPO):
    !git -C {REPO} pull -q
else:
    !git clone -q https://github.com/thebnbrkr/grassman.git {REPO}
%cd {REPO}
!git log --oneline -1
!pip install -q datasets transformers

Only if you are **resuming this notebook** in a new runtime: set `RESTORE = True` to upload the `results.zip` it downloaded earlier, so finished runs are skipped. On a first run leave it `False`.

In [ ]:
RESTORE = False
if RESTORE:
    from google.colab import files
    for name in files.upload():
        !cd /content && unzip -o -q "{name}"
!ls -la {OUTDIR}

Unit tests (about a minute). Stop if anything fails.

In [ ]:
!python -m pytest -q

## 1. Same numbers?

Differences of about 1e-5 or less are float32 rounding; anything larger is a bug.

In [ ]:
import torch
from grassman.bench import fast_vs_reference, throughput, set_reference
dev = torch.device('cuda')
for r in fast_vs_reference(dev):
    print(f"{r['kind']:<8} normalize={str(r['normalize']):<5}  max output diff {r['max_out_diff']:.2e}   "
          f"max gradient diff {r['max_grad_diff']:.2e}")

## 2. How much faster? (training tokens/s, batch 32 × 128, bf16)

In [ ]:
from grassman.model import Config
results = {}
for m in ['grassmann', 'sym', 'outer', 'concat', 'concat32', 'g-select', 'g-dilated', 'g-both', 'g-both-angle',
          'hyena', 'swa', 'attention']:
    cfg = Config(mixer=m)
    fast = throughput(cfg, dev)
    ref = throughput(cfg, dev, reference=True) if m in ('grassmann', 'sym', 'outer', 'concat', 'concat32', 'g-dilated') else None
    results[m] = {'fast': fast, 'paper_order': ref}
    print(f"{m:<13} fast {fast:>10,.0f} tok/s" + (f"   paper order {ref:>10,.0f}   speed-up {fast / ref:4.1f}x" if ref else ''))
json.dump({'gpu': torch.cuda.get_device_name(), 'tok_per_s': results}, open(RUNS, 'w'), indent=1)

## 3. Where does the time go?

Top GPU operations over 10 training steps. If the same operations dominate for `attention` and `grassmann`, the cost is in the shared parts (embedding, output layer, loss, optimizer), not the mixer.

In [ ]:
from torch.profiler import profile, ProfilerActivity
from grassman.model import LM
for m in ['attention', 'grassmann']:
    torch.manual_seed(0)
    model = LM(Config(mixer=m)).to(dev)
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    x = torch.randint(0, 30522, (32, 128), device=dev)
    def step():
        with torch.autocast('cuda', dtype=torch.bfloat16):
            _, loss = model(x, x)
        opt.zero_grad(set_to_none=True); loss.backward(); opt.step()
    for _ in range(5):
        step()
    torch.cuda.synchronize()
    with profile(activities=[ProfilerActivity.CPU, ProfilerActivity.CUDA]) as prof:
        for _ in range(10):
            step()
        torch.cuda.synchronize()
    print(f'===== {m} =====')
    print(prof.key_averages().table(sort_by='cuda_time_total', row_limit=15))

Download the results before the runtime is deleted.

In [ ]:
!cd /content && zip -qr results.zip results
from google.colab import files
files.download('/content/results.zip')